# Example 3: Real Read Quality — Streaming a Public Illumina Run



Stream the first 100 reads of a real public Illumina run (SRR2584863) via ENA and compute real Phred quality statistics.



In [2]:
# Stream just the first 100 reads from a real public Illumina run via ENA — no full download
import gzip
import itertools
import os
import requests

os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

url = "https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR258/003/SRR2584863/SRR2584863_1.fastq.gz"
n_reads = 100

response = requests.get(url, stream=True)
response.raise_for_status()
with gzip.GzipFile(fileobj=response.raw) as gz:
    lines = list(itertools.islice(gz, n_reads * 4))  # 4 lines per FASTQ read 
response.close()  # stop the connection now — the rest of the file is never fetched


# Quality lines are raw ASCII bytes — iterating bytes gives Phred+33 scores directly
def mean_phred(qual_line):
    return sum(c - 33 for c in qual_line) / len(qual_line)

qualities = [mean_phred(lines[i + 3].rstrip()) for i in range(0, len(lines), 4)]
read_length = len(lines[1].rstrip())
passing_q20 = sum(q >= 20 for q in qualities)

summary = (
    f"Reads sampled: {len(qualities)}\n"
    f"Read length: {read_length} bp\n"
    f"Mean Phred quality: {sum(qualities)/len(qualities):.1f}\n"
    f"Reads with mean quality >= Q20: {passing_q20}/{len(qualities)}\n"
)
print(summary)
with open("results/example3_fastq_quality.txt", "w") as f:
    f.write(summary)
print("Saved: results/example3_fastq_quality.txt")


Reads sampled: 100
Read length: 150 bp
Mean Phred quality: 31.4
Reads with mean quality >= Q20: 95/100

Saved: results/example3_fastq_quality.txt
